# Setup


In [ ]:
DATA_SOURCES = ["tag"]
ITERATION_LIMIT = 15
curr_iteration = -1

## Pneuma-Seeker-Specific


In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import time

from torch.backends import cudnn

# enforce more deterministic behavior
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

import pandas as pd

sys.path.append("../..")
sys.path.append("..")

from pneuma_seeker.core.chat_interface import ChatInterface
from pneuma_seeker.model.interface.impl.openai_llm import OpenAILLM
from pneuma_seeker.core.ir_system.data_model import convert_multi_retriever_results_to_str
from pneuma_seeker.model.llm_message import LLMMessage, Role
from pneuma_seeker.core.conductor.state import InformationNeedState
from pneuma_seeker.core.ir_system.data_model import AbstractDocument, RetrieverType
from pneuma_seeker.utils.config import Config
from pneuma_seeker.utils.logger import setup_logger

In [ ]:
llm_path = "o4-mini"
embed_model_path = "../model/weight/bge-base"

chat_interface = ChatInterface(
    llm_path,
    embed_model_path,
    "user_id",
    "chat_id",
    DATA_SOURCES,
    False,
    env_path="../../../.env",
)
config = Config("../../../.env")
logger = setup_logger(log_path=".")
gpt = OpenAILLM("gpt-4o", config, logger)

## Benchmark-Specific


In [ ]:
benchmark = pd.read_csv("../../../benchmark/sources/TAG/tag_queries.csv")

In [ ]:
def get_formatted_system_output(
    system_output: str,
    state: InformationNeedState,
    curr_retrieval_results: dict[RetrieverType, list[AbstractDocument]],
):
    return f"""SYSTEM OUTPUT:
```{system_output}```

STATE:
```{state}```

RETRIEVED DATA BY THE SYSTEM:
```{convert_multi_retriever_results_to_str(curr_retrieval_results)}```
"""

In [ ]:
def get_prompt_to_llm(question: str) -> str:
    """
    Returns a goal-focused system prompt for an LLM that will interact with
    a data-assistant system to produce a specific answer efficiently.
    """
    return f"""You are an expert analyst using a data assistant system to answer a specific question.
The system expresses its understanding as:
- a set of target schemas (tables it thinks are relevant)
- a list of SQL statements which, if run sequentially on these schemas, should produce an answer.

Your job is to:
1. Evaluate whether the system's representation (schemas + SQL) correctly matches the question.
2. If it does, confirm and proceed.
3. If it doesn't, refine your instructions or clarify requirements so the system aligns with your goal.

**Important:** Your ultimate objective is to correctly and efficiently answer this question:
```

{question}

```

Guidelines for your responses:
- Act like a domain expert (precise, critical, but not verbose).
- If the system misinterprets your need, correct it directly.
- Keep focus on the main question — only explore side ideas if they help clarify or validate the path to the answer.
- Speak to the system as if you are giving it instructions or feedback — do not roleplay with humans or produce narrative explanations.

Continue the conversation from here, giving your next message to the system:

YOU: {question}
""".strip()

# Interaction


In [ ]:
def get_system_output(
    chat_interface: ChatInterface,
    chat_messages: list[LLMMessage],
    external_data_paths: list[str] = [],
):
    start = time.time()
    system_output = ""
    for log_message in chat_interface.process_user_input(
        chat_messages, external_data_paths
    ):
        if log_message.startswith("LOG") or log_message.startswith("DONE"):
            continue
        system_output += log_message
    end = time.time()
    print(f"===> Responding in {end-start:.2f} seconds: {system_output}")
    return system_output

In [ ]:
curr_user_prompt = benchmark["Query"][4]
gpt_messages = [
    LLMMessage(role=Role.SYSTEM.value, content=get_prompt_to_llm(curr_user_prompt))
]
chat_messages: list[LLMMessage] = [
    LLMMessage(role=Role.USER.value, content=curr_user_prompt)
]
print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")

In [ ]:
curr_iteration += 1
system_output = get_system_output(chat_interface, chat_messages)
format_to_gpt = get_formatted_system_output(
    system_output,
    chat_interface.conductor.info_need_state,
    chat_interface.conductor.current_retrieval_results,
)
print(system_output)
chat_messages.append(
    LLMMessage(
        role=Role.ASSISTANT.value,
        content=system_output,
    )
)

if curr_iteration == 0:
    gpt_messages[0]["content"] += f"\n{format_to_gpt}"
else:
    gpt_messages.append(LLMMessage(role=Role.USER.value, content=format_to_gpt))

updated_user_prompt = gpt.chat(gpt_messages, LLMOption(temperature=0))
gpt_messages.append(LLMMessage(role=Role.ASSISTANT.value, content=updated_user_prompt))

if updated_user_prompt.startswith("YOU:"):
    updated_user_prompt = updated_user_prompt[4:]
    updated_user_prompt = updated_user_prompt.strip()
curr_user_prompt = updated_user_prompt

print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")